# NDLEA Event-Level Drug Enforcement Extractor (v2)

One row per **incident** (not per article). Publication date is metadata only.

**Pipeline (each cell runs/debugs on its own):**
listing scraper → article scraper → vocabularies → date parser → clause splitter → enforcement filter → drug/quantity extractor → arrest/state/location extractor → event builder → consolidation → validation → export

**Before scraping at scale:** run the robots.txt cell, keep the request delay, and consider dropping any personal names before publishing a public dashboard (this notebook never stores suspect names).

**Design rules baked in (from your spec):**
1. Arrest info is clause-local. Never broadcast an article-level total.
2. `mg` values ≤ 1000 are treated as drug *strength* (e.g. tramadol 225mg), never as a quantity.
3. A quantity is linked to a drug only via explicit local patterns and a ~100-character distance gate; otherwise `drug_type` stays blank and confidence is Low.
4. Dates: never the publication date by default; any date after the publication date is rejected (set to NULL).
5. Near-duplicates (same URL + drug + unit + state + quantity within 1%) are consolidated.
6. Non-enforcement articles (MoU, speeches, recruitment, WADA-only, etc.) are excluded.
7. `quantity_kg > 5000` is flagged Low unless the clause mentions a farm/plantation/hectares/destruction.
8. `validate()` **raises** on hard errors and `export_csv()` refuses to write until they are fixed.

**Two deliberate judgement calls (documented so you can override them):**
- *List continuations:* in "arrested X with 160g Tramadol; 80 pills of Diazepam; 38g Exol", the later items have no verb of their own. They inherit the arrest info from the clause that governs the list (marked Medium, and given the same `arrest_group_id` so you can avoid counting the same arrested person several times when summing arrests).
- *Parcels + kilograms:* "4,000 parcels of Loud ... weighing 2,326 kilograms" is one seizure with two measurements. It becomes one row keeping the kg figure (Medium).

**Arrest totals:** one arrested person can appear on several seizure rows. When summing arrests, count each `arrest_group_id` once.


## Setup & config

In [ ]:
!pip install -q requests beautifulsoup4 lxml python-dateutil

import re, time, random, urllib.parse
from datetime import datetime, timedelta
from urllib.robotparser import RobotFileParser

import requests
import numpy as np
import pandas as pd
from bs4 import BeautifulSoup
from dateutil import parser as dateparser

BASE_URL = "https://ndlea.gov.ng"
NEWS_LIST_URL = f"{BASE_URL}/news"
ARTICLE_URL_PREFIX = f"{BASE_URL}/blog/"
HEADERS = {"User-Agent": "Mozilla/5.0 (research-scraper; contact: YOUR_EMAIL_HERE)"}
REQUEST_DELAY_RANGE = (2.0, 4.5)   # be polite; do not remove

def polite_sleep():
    time.sleep(random.uniform(*REQUEST_DELAY_RANGE))

## robots.txt check (run first)

In [ ]:
rp = RobotFileParser()
rp.set_url(f"{BASE_URL}/robots.txt")
try:
    rp.read()
    print("Can fetch /news  :", rp.can_fetch(HEADERS["User-Agent"], NEWS_LIST_URL))
    print("Can fetch /blog/*:", rp.can_fetch(HEADERS["User-Agent"], ARTICLE_URL_PREFIX + "example"))
except Exception as e:
    print("Could not read robots.txt automatically:", e)
try:
    print(requests.get(f"{BASE_URL}/robots.txt", headers=HEADERS, timeout=15).text)
except Exception as e:
    print("Could not fetch robots.txt content:", e)

## Diagnostic - inspect real HTML before trusting the selectors below

In [ ]:
def debug_fetch_raw(url):
    r = requests.get(url, headers=HEADERS, timeout=20)
    r.raise_for_status()
    return r.text

# Replace with any current article URL, then look at the tags around the title, date and body:
# print(debug_fetch_raw(ARTICLE_URL_PREFIX + "some-article-slug")[:4000])

## Listing scraper (collect article URLs)

In [ ]:
def get_article_links_from_page(page_url):
    r = requests.get(page_url, headers=HEADERS, timeout=20)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "lxml")
    links, seen = [], set()
    for a in soup.find_all("a", href=True):
        if "/blog/" in a["href"]:
            url = urllib.parse.urljoin(BASE_URL, a["href"])
            if url not in seen:
                seen.add(url)
                links.append({"title": a.get_text(strip=True) or None, "url": url})
    return links

def get_all_article_links(max_pages=5, page_param_style="query"):
    """page_param_style 'query' -> ?page=N ; 'path' -> /page/N. Confirm the real pattern first."""
    all_links = {}
    for n in range(1, max_pages + 1):
        if n == 1:
            page_url = NEWS_LIST_URL
        elif page_param_style == "query":
            page_url = f"{NEWS_LIST_URL}?page={n}"
        else:
            page_url = f"{NEWS_LIST_URL}/page/{n}"
        print("Listing page", n, page_url)
        try:
            found = get_article_links_from_page(page_url)
        except Exception as e:
            print("  failed:", e); break
        new = [x for x in found if x["url"] not in all_links]
        for x in new: all_links[x["url"]] = x
        print(f"  {len(found)} links, {len(new)} new")
        if not new: break
        polite_sleep()
    return list(all_links.values())

## Article scraper (title, publication date, body)

In [ ]:
def fetch_article(url):
    r = requests.get(url, headers=HEADERS, timeout=20)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "lxml")

    h1 = soup.find("h1")
    title = h1.get_text(strip=True) if h1 else None

    # Header line looks like "May 10, 2026 - 10:25" -> metadata only
    page_text = soup.get_text("\n", strip=True)
    m = re.search(r"([A-Z][a-z]{2,8}\s+\d{1,2},?\s+\d{4})\s*-\s*\d{1,2}:\d{2}", page_text[:3000])
    pub_raw = m.group(1) if m else None
    try:
        pub_dt = dateparser.parse(pub_raw) if pub_raw else None
    except Exception:
        pub_dt = None

    body = None
    for kw in ({"class_": re.compile(r"(post|blog|entry|article).*(content|body|details)", re.I)},
               {"class_": re.compile(r"content", re.I)}):
        cand = soup.find("div", **kw)
        if cand and len(cand.get_text(strip=True)) > 200:
            body = cand; break
    if body is None:
        best, best_len = None, 0
        for div in soup.find_all("div"):
            n = sum(len(p.get_text(strip=True)) for p in div.find_all("p", recursive=False))
            if n > best_len: best, best_len = div, n
        body = best
    body_text = "\n".join(p.get_text(" ", strip=True) for p in body.find_all("p") if p.get_text(strip=True)) if body else ""

    return {"title": title, "publication_date": pub_dt, "publication_date_raw": pub_raw,
            "body_text": body_text, "source_url": url}

## Vocabularies - drugs, units, states

In [ ]:
DRUG_CANON = {   # alias -> canonical name reported in the dataset
    "cannabis sativa": "cannabis", "indian hemp": "cannabis", "marijuana": "cannabis",
    "canadian loud": "loud", "crystal meth": "methamphetamine", "meth": "methamphetamine",
    "exol 5": "exol-5", "exol": "exol-5", "codeine syrup": "codeine",
}
DRUG_VOCAB = sorted([
    "cannabis sativa", "cannabis oil", "cannabis seeds", "cannabis", "indian hemp", "marijuana",
    "skunk", "canadian loud", "loud", "colorado", "arizona", "monkey tail", "skuchies",
    "cocaine", "heroin", "methamphetamine", "crystal meth", "meth",
    "tramadol", "codeine syrup", "codeine", "diazepam", "rohypnol",
    "exol-5", "exol 5", "exol", "mdma", "ecstasy", "ketamine", "captagon",
    "tapentadol", "pentazocine", "bromazepam", "psychotropic substances", "narcotics",
], key=len, reverse=True)
# Generic terms often used as an appositive for a named strain ("Loud, a strain of imported cannabis").
GENERIC_DRUGS = {"cannabis", "psychotropic substances", "narcotics"}
DRUG_RE = re.compile(r"\b(" + "|".join(re.escape(d) for d in DRUG_VOCAB) + r")\b", re.IGNORECASE)

def canon(drug):
    d = drug.lower().strip()
    return DRUG_CANON.get(d, d)

MASS_UNITS = {
    "kg": 1.0, "kgs": 1.0, "kilogram": 1.0, "kilograms": 1.0, "kilogramme": 1.0, "kilogrammes": 1.0,
    "g": 1e-3, "gram": 1e-3, "grams": 1e-3, "gramme": 1e-3, "grammes": 1e-3,
    "mg": 1e-6, "milligram": 1e-6, "milligrams": 1e-6,
    "tonne": 1000.0, "tonnes": 1000.0, "ton": 1000.0, "tons": 1000.0,
}
MG_UNITS = {"mg", "milligram", "milligrams"}
NON_MASS_UNITS = {
    "tablet", "tablets", "pill", "pills", "capsule", "capsules", "bottle", "bottles",
    "sachet", "sachets", "wrap", "wraps", "parcel", "parcels", "block", "blocks",
    "carton", "cartons",
    "litre", "litres", "liter", "liters", "ampoule", "ampoules",   # extras seen in real articles
}
ALL_UNITS = set(MASS_UNITS) | NON_MASS_UNITS
UNIT_ALT = "|".join(sorted((re.escape(u) for u in ALL_UNITS), key=len, reverse=True))
QTY = r"\d{1,3}(?:,\d{3})+(?:\.\d+)?|\d+(?:\.\d+)?"

# "<qty> <unit> [of] <up to 4 words>"  (drug is looked up inside the words)
TEMPLATE1_RE = re.compile(
    rf"(?<![\w.,])(?P<qty>{QTY})\s*(?P<unit>{UNIT_ALT})\b\s*(?:of\s+)?"
    rf"(?P<gap>[A-Za-z][A-Za-z\-]*(?:\s+[A-Za-z][A-Za-z\-]*){{0,3}})", re.IGNORECASE)
# "weighing <qty> <unit>"
WEIGHING_RE = re.compile(
    rf"(?:weighing|weighed|totalling|totaling|amounting\s+to)\s+(?:about\s+|approximately\s+|over\s+)?"
    rf"(?P<qty>{QTY})\s*(?P<unit>{UNIT_ALT})\b", re.IGNORECASE)
QTY_UNIT_ANY_RE = re.compile(rf"(?<![\w.,])(?:{QTY})\s*(?:{UNIT_ALT})\b", re.IGNORECASE)

DISTANCE_GATE = 100   # max characters between a drug mention and its "weighing" quantity

STATE_ALIASES = {"abuja": "FCT", "fct": "FCT", "federal capital territory": "FCT"}
NIGERIAN_STATES = ["Abia","Adamawa","Akwa Ibom","Anambra","Bauchi","Bayelsa","Benue","Borno","Cross River",
    "Delta","Ebonyi","Edo","Ekiti","Enugu","Gombe","Imo","Jigawa","Kaduna","Kano","Katsina","Kebbi","Kogi",
    "Kwara","Lagos","Nasarawa","Niger","Ogun","Ondo","Osun","Oyo","Plateau","Rivers","Sokoto","Taraba",
    "Yobe","Zamfara","FCT","Abuja","Federal Capital Territory"]
STATE_RE = re.compile(r"\b(" + "|".join(re.escape(s) for s in sorted(NIGERIAN_STATES, key=len, reverse=True)) + r")\b", re.IGNORECASE)

## Date parsing (incident dates come from the event text only)

In [ ]:
MONTH_RE = r"(January|February|March|April|May|June|July|August|September|October|November|December)"
WEEKDAY = r"(?:(?:Mon|Tues?|Wed(?:nes)?|Thu(?:rs)?|Fri|Sat(?:ur)?|Sun)(?:day)?\s+)?"
DATE_FULL_RE = re.compile(rf"\b{WEEKDAY}(\d{{1,2}})(?:st|nd|rd|th)?\s+(?:of\s+)?{MONTH_RE},?\s+(\d{{4}})\b", re.I)
DATE_NOYEAR_RE = re.compile(rf"\b{WEEKDAY}(\d{{1,2}})(?:st|nd|rd|th)?\s+(?:of\s+)?{MONTH_RE}\b(?!,?\s+\d{{4}})", re.I)
SAME_DAY_RE = re.compile(r"\b(?:on\s+)?the\s+same\s+day\b|\bsame\s+day\b", re.I)
MONTHS = {m: i + 1 for i, m in enumerate(["january","february","march","april","may","june","july",
                                          "august","september","october","november","december"])}

def parse_incident_date(text, publication_date, last_known_date):
    """Returns (datetime|None, note). Never returns a date after publication_date."""
    def guard(d, note):
        if publication_date is not None and d.date() > publication_date.date():
            return None, "future_date_rejected"
        return d, note

    m = DATE_FULL_RE.search(text)
    if m:
        try:
            return guard(datetime(int(m.group(3)), MONTHS[m.group(2).lower()], int(m.group(1))), "explicit_full_date")
        except ValueError:
            return None, "invalid_date"

    m = DATE_NOYEAR_RE.search(text)
    if m:
        if publication_date is None:
            return None, "no_year_no_reference"
        day, month = int(m.group(1)), MONTHS[m.group(2).lower()]
        pub_d = publication_date.date()
        for year in (publication_date.year, publication_date.year - 1):
            try:
                cand = datetime(year, month, day)
            except ValueError:
                return None, "invalid_date"
            if cand.date() <= pub_d:
                # Previous-year fallback is only unambiguous for a recent event (e.g. a December raid in a January article).
                if year != publication_date.year and (pub_d - cand.date()).days > 60:
                    return None, "future_date_rejected"
                return cand, "explicit_no_year_inferred"
        return None, "future_date_rejected"

    if SAME_DAY_RE.search(text) and last_known_date is not None:
        return last_known_date, "same_day_reference"
    return None, "no_date_found"

def distinct_dates_in(text):
    return {(m.group(1), m.group(2).lower()) for m in DATE_NOYEAR_RE.finditer(text)} | \
           {(m.group(1), m.group(2).lower(), m.group(3)) for m in DATE_FULL_RE.finditer(text)}

## Paragraph / sentence / clause splitter

In [ ]:
SENT_SPLIT_RE = re.compile(r"(?<!\bMr)(?<!\bMrs)(?<!\bDr)(?<!\bProf)(?<!\bHon)(?<!\bCol)(?<!\bNo)(?<!\bSt)(?<!\bLt)(?<!\bGen)"
                           r"[.!?](?=\s+[A-Z0-9\"\u201c])")
CLAUSE_SPLIT_RE = re.compile(r";|\s+while\s+|\s+as\s+(?!well\b)", re.I)

def split_paragraphs(text):
    return [p.strip() for p in re.split(r"\n+", text) if p.strip()]

def split_sentences(paragraph):
    parts = SENT_SPLIT_RE.split(re.sub(r"\s+", " ", paragraph))
    return [p.strip() for p in parts if p.strip()]

def _has_event_content(fragment):
    return bool(DRUG_RE.search(fragment) and QTY_UNIT_ANY_RE.search(fragment))

def split_clauses(sentence):
    """Split on ';' / 'while' / 'as' - but only where the right-hand side carries its own drug+quantity.
    That keeps suspect-name lists ('A; B and C') and 'arrested as they ...' together with their event."""
    bounds = list(CLAUSE_SPLIT_RE.finditer(sentence))
    cuts = []
    for i, b in enumerate(bounds):
        right_end = bounds[i + 1].start() if i + 1 < len(bounds) else len(sentence)
        if _has_event_content(sentence[b.end():right_end]):
            cuts.append(b)
    pieces, start = [], 0
    for b in cuts:
        pieces.append(sentence[start:b.start()]); start = b.end()
    pieces.append(sentence[start:])
    return [p.strip(" ,;") for p in pieces if p.strip(" ,;")]

## Non-enforcement article filter

In [ ]:
NON_ENFORCEMENT_TITLE_RE = re.compile(
    r"recruit|screening|interview|training|workshop|conference|summit|symposium|keynote|speech|declaration|"
    r"appoint|promot|decorat|award|public notice|memorandum|\bmou\b|partnership|cooperation|collaborat|"
    r"sensiti[sz]ation|enlightenment|advocacy|wada|graduation|passing out|anniversary|courtesy|visit|"
    r"handover|inaugurat|commission(?:s|ed|ing)?\b|donat", re.I)
ENFORCEMENT_VERB_RE = re.compile(r"\b(arrest(?:ed)?|nabbed|apprehended|seiz(?:ed|ure)|intercept(?:ed|ion)|recover(?:ed|y)|raid(?:ed)?|busted|dismantl(?:ed|ing)|destroy(?:ed)?|uncovered|discovered|found)\b", re.I)

def concrete_enforcement_sentences(body):
    """Sentences with a verb + drug + quantity (candidate real incidents)."""
    out = []
    for para in split_paragraphs(body):
        for s in split_sentences(para):
            if ENFORCEMENT_VERB_RE.search(s) and DRUG_RE.search(s) and QTY_UNIT_ANY_RE.search(s):
                out.append(s)
    return out

def is_enforcement_article(title, body):
    concrete = concrete_enforcement_sentences(body)
    if not concrete:
        return False
    if NON_ENFORCEMENT_TITLE_RE.search(title or ""):
        # Speeches / MoUs / campaigns often quote statistics. Require at least one dated, concrete incident.
        return any(DATE_FULL_RE.search(s) or DATE_NOYEAR_RE.search(s) for s in concrete)
    return True

## Drug <-> quantity extractor (tight, clause-local association)

In [ ]:
def _pick_backward_drug(preceding):
    """Nearest drug mention before a 'weighing' quantity. Prefers a specific strain over a generic
    descriptor that sits between them. Returns (canonical_drug, distance_in_chars) or (None, None)."""
    matches = list(DRUG_RE.finditer(preceding))
    if not matches:
        return None, None
    chosen = matches[-1]
    if canon(chosen.group(1)) in GENERIC_DRUGS and len(matches) > 1:
        for m in reversed(matches[:-1]):
            if canon(m.group(1)) not in GENERIC_DRUGS:
                chosen = m; break
    return canon(chosen.group(1)), len(preceding) - chosen.end()

def extract_drug_quantity_pairs(clause):
    """Returns list of dicts: drug_type, quantity, unit, quantity_kg, confidence, note.
    Never combines every drug with every quantity: each quantity is tied to ONE drug by a local pattern."""
    results, claimed = [], []
    overlaps = lambda a, b: any(not (b <= s or a >= e) for s, e in claimed)

    # Pattern A: "<qty> <unit> [of] <drug>"  (also handles "A and B" coordination, one match per quantity)
    for m in TEMPLATE1_RE.finditer(clause):
        qty = float(m.group("qty").replace(",", ""))
        unit = m.group("unit").lower()
        if unit in MG_UNITS and qty <= 1000:
            continue                                   # tablet STRENGTH (e.g. tramadol 225mg), not a quantity
        gap = m.group("gap")
        dm = DRUG_RE.search(gap)
        if not dm or dm.start() > 40:                  # distance gate
            continue
        span = (m.start("qty"), m.start("gap") + dm.end())
        if overlaps(*span):
            continue
        claimed.append(span)
        results.append({"drug_type": canon(dm.group(1)), "quantity": qty, "unit": unit,
                        "quantity_kg": round(qty * MASS_UNITS[unit], 6) if unit in MASS_UNITS else None,
                        "confidence": "High" if dm.start() <= 20 else "Medium", "note": "qty_unit_of_drug"})

    # Pattern B: "<drug> ... weighing <qty> <unit>"
    for m in WEIGHING_RE.finditer(clause):
        qty = float(m.group("qty").replace(",", ""))
        unit = m.group("unit").lower()
        if unit in MG_UNITS and qty <= 1000:
            continue
        if overlaps(m.start("qty"), m.end("unit")):
            continue
        drug, dist = _pick_backward_drug(clause[:m.start()])
        if drug is None or dist > DISTANCE_GATE:
            results.append({"drug_type": None, "quantity": qty, "unit": unit,
                            "quantity_kg": round(qty * MASS_UNITS[unit], 6) if unit in MASS_UNITS else None,
                            "confidence": "Low", "note": "weighing_no_drug_within_gate"})
            continue
        kg = round(qty * MASS_UNITS[unit], 6) if unit in MASS_UNITS else None
        for ex in results:                            # same seizure counted in parcels AND kg -> one row, keep kg
            if ex["drug_type"] == drug and ex["unit"] in NON_MASS_UNITS and unit in MASS_UNITS:
                ex.update(quantity=qty, unit=unit, quantity_kg=kg, confidence="Medium",
                          note="merged_count_and_weight_kept_kg")
                break
        else:
            results.append({"drug_type": drug, "quantity": qty, "unit": unit, "quantity_kg": kg,
                            "confidence": "Medium", "note": "drug_weighing_qty"})
    return results

## Arrest / state / location / incident-type extractors (clause-local)

In [ ]:
NUM_WORDS = {"a": 1, "an": 1, "one": 1, "two": 2, "three": 3, "four": 4, "five": 5, "six": 6, "seven": 7,
             "eight": 8, "nine": 9, "ten": 10, "eleven": 11, "twelve": 12}
ARREST_WORDS = r"arrested|nabbed|apprehended|detained|taken\s+into\s+custody"
ARREST_TERM_RE = re.compile(rf"\b({ARREST_WORDS}|arrest\s+of)\b", re.I)
NO_ARREST_RE = re.compile(r"\bno\s+(?:one\s+was\s+|suspects?\s+(?:was|were)\s+)?arrest(?:s|ed)?\b|\bwithout\s+(?:any\s+)?arrests?\b|"
                          r"\bnobody\s+was\s+arrested\b", re.I)
PERSON_NOUNS = (r"(?:other\s+)?(?:suspects?|persons?|people|men|women|traffickers?|dealers?|individuals?|pushers?|"
                r"couriers?|youths?|foreigners?|nationals?|PWDs?|members|drug\s+(?:suspects|traffickers|dealers|barons))")
COUNT_RE = re.compile(rf"\b(?P<num>\d{{1,4}}|{'|'.join(NUM_WORDS)})\s+(?:\d{{1,2}}-year-old\s+)?{PERSON_NOUNS}\b", re.I)
NAME_TOKEN = r"[A-Z][a-z]+(?:[-'][A-Za-z]+)?"
NAME = rf"{NAME_TOKEN}(?:\s+{NAME_TOKEN}){{1,2}}"
NAME_BEFORE_RE = re.compile(rf"(?P<name>{NAME})(?:,\s*\d{{1,2}},?)?\s+(?:was\s+|were\s+)?(?:{ARREST_WORDS})")
NAME_AFTER_RE = re.compile(rf"(?:{ARREST_WORDS})\s+(?:another\s+|a\s+)?(?:[A-Z]{{2,5}}\s+)?(?P<name>{NAME})(?:,\s*\d{{1,2}}|\s+with\b|\s+over\b|\s+in\b|\s+at\b)")
NAME_BLACKLIST = {"national","drug","law","enforcement","agency","command","operations","operatives","operative",
                  "unit","special","nigeria","nigerian","airport","commander","officers","officer","police","customs",
                  "service","state","federal","ndlea","marc","chairman","director","the","interdiction","patrol"}

def _valid_names(regex, text):
    names = set()
    for m in regex.finditer(text):
        n = m.group("name")
        toks = [t.lower() for t in n.split()]
        if any(t in NAME_BLACKLIST for t in toks) or STATE_RE.fullmatch(n):
            continue
        names.add(n)
    return names

def extract_arrest_info(clause):
    """(arrest_count|None, arrest_status). Clause-local only; count is None unless stated; never 0."""
    if NO_ARREST_RE.search(clause):
        return None, "No"
    terms = list(ARREST_TERM_RE.finditer(clause))
    if not terms:
        return None, "Not stated"
    for m in COUNT_RE.finditer(clause):
        if min(abs(m.start() - t.start()) for t in terms) <= 120:
            raw = m.group("num").lower()
            return float(NUM_WORDS.get(raw) or int(raw)), "Yes"
    names = _valid_names(NAME_BEFORE_RE, clause) | _valid_names(NAME_AFTER_RE, clause)
    if names:
        return float(len(names)), "Yes"
    return None, "Yes"       # arrest language present, count genuinely unstated

def extract_state(text, anchor=None):
    """(state|None, ambiguous). One state only; if several are named, pick the nearest to the anchor and flag it."""
    found = [(m.start(), STATE_ALIASES.get(m.group(1).lower(), m.group(1).title())) for m in STATE_RE.finditer(text)]
    if not found:
        return None, False
    distinct = {s for _, s in found}
    if len(distinct) == 1:
        return found[0][1], False
    if anchor is None:
        return None, True
    return min(found, key=lambda x: abs(x[0] - anchor))[1], True

_LOC_RE = re.compile(r"\b(?:at|in|along|around|near|within|inside)\s+(?:the\s+)?((?:[A-Z][\w'/\-]*)(?:\s+[A-Z][\w'/\-]*){0,3})")
_LOC_JUNK = re.compile(r"\b(?:Mon|Tues?|Wednes|Thurs?|Fri|Satur|Sun)day\b|\b(?:January|February|March|April|May|June|July|August|"
                       r"September|October|November|December)\b|\b\d{1,2}(?:st|nd|rd|th)\b|\b\d{4}\b|\bState\b|"
                       r"\b(?:NDLEA|National|Agency|Command|Operatives)\b", re.I)
def extract_location(clause):
    """Best-effort place name from the same clause (dates / 'State' stripped). None if unsure."""
    for m in _LOC_RE.finditer(clause):
        loc = _LOC_JUNK.sub("", m.group(1)).strip(" ,/-")
        if loc and not STATE_RE.fullmatch(loc):
            return loc
    return None

_FARM_RE = re.compile(r"\b(farm|farms|plantation|hectares?|destroy(?:ed)?|destruction|eradicat\w*)\b", re.I)
_LAB_RE = re.compile(r"\b(laborator(?:y|ies)|clandestine|dismantl\w*|meth\s+lab)\b", re.I)
_INTERDICT_RE = re.compile(r"\b(intercept\w*|checkpoint|patrol\w*|airport|seaport|port|border|bus|vehicle|truck|luggage|courier|parcel)\b", re.I)
def classify_incident_type(clause, arrest_status):
    if _LAB_RE.search(clause):      return "lab_dismantling"
    if _FARM_RE.search(clause):     return "farm_destruction"
    if arrest_status == "Yes":      return "arrest"
    if _INTERDICT_RE.search(clause): return "interdiction"
    return "seizure"

## Event builder (article -> clause-level event rows)

In [ ]:
LIST_ITEM_RE = re.compile(r"^(?:and\s+)?\d")            # fragment starts with a quantity: elliptical list item
OWN_VERB_RE = re.compile(r"\b(recovered|intercepted|seized|found|discovered|arrested|nabbed|apprehended|detained|raided|uncovered)\b", re.I)
ALLOWED_EQ_PUB_NOTES = {"explicit_full_date", "explicit_no_year_inferred", "inherited_from_sentence", "same_day_reference"}

def _qty_anchor(clause):
    m = QTY_UNIT_ANY_RE.search(clause)
    return m.start() if m else None

def process_article(article, article_idx=0):
    title, pub, body = article.get("title"), article.get("publication_date"), article.get("body_text", "")
    if not is_enforcement_article(title, body):
        return []
    events, last_known, group_n = [], None, 0

    for para in split_paragraphs(body):
        for sentence in split_sentences(para):
            sent_dates = distinct_dates_in(sentence)
            sent_date, sent_note = parse_incident_date(sentence, pub, last_known)
            sent_states = {STATE_ALIASES.get(m.group(1).lower(), m.group(1).title()) for m in STATE_RE.finditer(sentence)}
            run_date = run_state = None
            prev_arrest, prev_group = (None, "Not stated"), None
            sent_events = []

            for clause in split_clauses(sentence):
                # ---- date (clause first, then running value inside this sentence, then the sentence's single date)
                c_date, c_note = parse_incident_date(clause, pub, last_known)
                if c_note in ("explicit_full_date", "explicit_no_year_inferred"):
                    date, note, run_date, last_known = c_date, c_note, c_date, c_date
                elif c_note == "same_day_reference":
                    date, note = c_date, c_note
                elif c_note in ("future_date_rejected", "invalid_date"):
                    date, note = None, c_note
                elif run_date is not None:
                    date, note = run_date, "inherited_from_sentence"
                elif len(sent_dates) == 1 and sent_date is not None:
                    date, note = sent_date, "inherited_from_sentence"
                else:
                    date, note = None, "no_date_found"

                # ---- state
                anchor = _qty_anchor(clause)
                state, amb = extract_state(clause, anchor)
                if state is not None:
                    run_state = state
                elif not amb:
                    state = run_state or (next(iter(sent_states)) if len(sent_states) == 1 else None)

                # ---- arrest (clause-local; list continuations inherit from the governing clause)
                count, status = extract_arrest_info(clause)
                group, inherited = None, False
                if status == "Yes":
                    group_n += 1; group = f"{article_idx}-{group_n}"
                    prev_arrest, prev_group = (count, status), group
                elif status == "No":
                    prev_arrest, prev_group = (None, "No"), None
                elif prev_arrest[1] == "Yes" and LIST_ITEM_RE.match(clause) and not OWN_VERB_RE.search(clause):
                    count, status = prev_arrest; group, inherited = prev_group, True

                pairs = extract_drug_quantity_pairs(clause)
                location = extract_location(clause) if pairs else None
                for p in pairs:
                    conf = p["confidence"]
                    if p["drug_type"] is None:
                        conf = "Low"
                    if date is None and conf == "High":
                        conf = "Medium"
                    if note == "same_day_reference" and conf == "High":
                        conf = "Medium"
                    if (amb or inherited) and conf == "High":
                        conf = "Medium"
                    if p["quantity_kg"] and p["quantity_kg"] > 5000 and not _FARM_RE.search(clause):
                        conf = "Low"
                    sent_events.append({
                        "incident_date": date.date().isoformat() if date else None,
                        "publication_date": pub.date().isoformat() if pub else None,
                        "article_title": title,
                        "incident_type": classify_incident_type(clause, status),
                        "drug_type": p["drug_type"], "quantity": p["quantity"], "unit": p["unit"],
                        "quantity_kg": p["quantity_kg"], "location": location, "state": state,
                        "arrest_count": count, "arrest_status": status,
                        "arrest_group_id": group, "source_url": article.get("source_url"),
                        "extraction_confidence": conf, "debug_clause": clause[:200],
                        "_clause_full": clause, "_date_note": note, "_note": p["note"], "_sentence": sentence,
                    })

            # single-event sentence: the sentence's own arrest language belongs to that event (not broadcasting)
            if len(sent_events) == 1 and sent_events[0]["arrest_status"] == "Not stated":
                s_count, s_status = extract_arrest_info(sentence)
                if s_status == "Yes":
                    group_n += 1
                    sent_events[0].update(arrest_count=s_count, arrest_status="Yes", arrest_group_id=f"{article_idx}-{group_n}")
                    if sent_events[0]["incident_type"] in ("seizure", "interdiction"):
                        sent_events[0]["incident_type"] = "arrest"
            events.extend(sent_events)
    return events

def build_dataframe(events):
    cols = ["incident_date","publication_date","article_title","incident_type","drug_type","quantity","unit",
            "quantity_kg","location","state","arrest_count","arrest_status","arrest_group_id","source_url",
            "extraction_confidence","debug_clause","_clause_full","_date_note","_note","_sentence"]
    df = pd.DataFrame(events, columns=cols)
    df["arrest_count"] = pd.to_numeric(df["arrest_count"], errors="coerce")   # NaN stays NaN, never 0
    return df

def run_pipeline(article_urls, verbose=True):
    events = []
    for i, url in enumerate(article_urls):
        if verbose: print(f"[{i+1}/{len(article_urls)}] {url}")
        try:
            art = fetch_article(url)
        except Exception as e:
            print("  FAILED:", e); continue
        events.extend(process_article(art, article_idx=i))
        polite_sleep()
    return build_dataframe(events)

## Post-extraction consolidation (deduplicate the same seizure described more than once)

In [ ]:
CONF_RANK = {"High": 3, "Medium": 2, "Low": 1}

def consolidate_events(df, tol=0.01):
    """Same source_url + drug + unit + state + quantity (within 1%) -> keep the best row.
    Best = has incident_date, then higher confidence, then coherent arrest info.
    Rows with two DIFFERENT known dates are separate incidents and are never merged."""
    if df.empty:
        return df.copy()
    d = df.copy()
    d["_score"] = (d["incident_date"].notna().astype(int) * 100
                   + d["extraction_confidence"].map(CONF_RANK).fillna(0).astype(int) * 10
                   + ((d["arrest_status"] == "Yes") & d["arrest_count"].notna()).astype(int))
    order = d.sort_values("_score", ascending=False, kind="stable").index
    kept, seen = [], {}
    for idx in order:
        r = d.loc[idx]
        key = (r["source_url"], r["drug_type"] if pd.notna(r["drug_type"]) else "", r["unit"],
               r["state"] if pd.notna(r["state"]) else "")
        dup = False
        for q0, dt0 in seen.get(key, []):
            same_qty = max(r["quantity"], q0) == 0 or abs(r["quantity"] - q0) / max(r["quantity"], q0) <= tol
            date_clash = pd.notna(r["incident_date"]) and pd.notna(dt0) and r["incident_date"] != dt0
            if same_qty and not date_clash:
                dup = True; break
        if not dup:
            kept.append(idx); seen.setdefault(key, []).append((r["quantity"], r["incident_date"]))
    return d.loc[sorted(kept)].drop(columns="_score")

## Validation suite (raises on hard errors; prints review lists)

In [ ]:
class ValidationError(Exception):
    pass

def validate(df, strict=True, verbose=True):
    hard = {}
    d = df.copy()
    pub = pd.to_datetime(d["publication_date"], errors="coerce")
    inc = pd.to_datetime(d["incident_date"], errors="coerce")

    hard["1_duplicates_after_consolidation"] = d.loc[~d.index.isin(consolidate_events(d).index)]
    hard["2_incident_eq_publication_without_explicit_date"] = d[(inc == pub) & inc.notna() & ~d["_date_note"].isin(ALLOWED_EQ_PUB_NOTES)]
    hard["3_incident_after_publication"] = d[(inc > pub)]
    g = d.groupby(["source_url", "_clause_full"])
    dup_idx = [i for _, grp in g if grp["drug_type"].nunique() > 1 and grp["quantity"].duplicated(keep=False).any()
               for i in grp.index[grp["quantity"].duplicated(keep=False)]]
    hard["4_same_quantity_across_different_drugs"] = d.loc[dup_idx]
    hard["5_multiple_states_in_one_row"] = d[d["state"].fillna("").str.contains(r"[,;/&]| and ")]
    hard["6_arrest_count_zero"] = d[d["arrest_count"] == 0]
    hard["7_negative_quantity"] = d[d["quantity"] < 0]
    hard["8_mg_strength_emitted_as_quantity"] = d[d["unit"].isin(MG_UNITS) & (d["quantity"] <= 1000)]
    hard["9_ambiguous_association_not_flagged"] = d[d["_note"].eq("weighing_no_drug_within_gate") & (d["drug_type"].notna() | (d["extraction_confidence"] != "Low"))]
    hard["10_arrest_count_without_arrest_yes"] = d[d["arrest_count"].notna() & (d["arrest_status"] != "Yes")]
    hard["11_bad_enum_values"] = d[~d["arrest_status"].isin(["Yes", "No", "Not stated"]) | ~d["extraction_confidence"].isin(["High", "Medium", "Low"])]
    hard["12_kg_for_non_mass_unit"] = d[d["quantity_kg"].notna() & ~d["unit"].isin(MASS_UNITS)]
    hard["13_huge_quantity_not_low_without_farm_context"] = d[(d["quantity_kg"] > 5000) & (d["extraction_confidence"] != "Low")
                                                              & ~d["_clause_full"].apply(lambda t: bool(_FARM_RE.search(t)))]
    failures = {k: v for k, v in hard.items() if len(v)}

    if verbose:
        print("=" * 70, f"\nRows: {len(d)}")
        for k in hard:
            print(f"  [{'FAIL' if k in failures else 'ok  '}] {k}: {len(hard[k])}")
        cols = ["article_title", "incident_date", "drug_type", "quantity", "unit", "state", "arrest_count", "arrest_status", "extraction_confidence"]
        print("\n-- 20 random sample rows --");                       print(d.sample(min(20, len(d)), random_state=1)[cols].to_string() if len(d) else "(none)")
        print("\n-- extraction_confidence != High --");               print(d[d["extraction_confidence"] != "High"][cols + ["debug_clause"]].to_string())
        print("\n-- incident_date is NULL --");                       print(d[d["incident_date"].isna()][cols + ["debug_clause"]].to_string())
        print("\n-- drug_type NULL but quantity exists --");          print(d[d["drug_type"].isna() & d["quantity"].notna()][cols + ["debug_clause"]].to_string())
        print("\n-- arrest_count filled AND status 'Not stated' (must be empty) --")
        print(d[d["arrest_count"].notna() & (d["arrest_status"] == "Not stated")][cols].to_string())
        print("\n-- quantity_kg > 5000 --");                          print(d[d["quantity_kg"] > 5000][cols + ["debug_clause"]].to_string())

    if strict and failures:
        raise ValidationError("Validation failed: " + ", ".join(f"{k} ({len(v)} rows)" for k, v in failures.items()))
    return failures

## Finalise + export (refuses to write if validation fails)

In [ ]:
EXPORT_COLUMNS = ["event_id","incident_date","publication_date","article_title","incident_type","drug_type",
                  "quantity","unit","quantity_kg","location","state","arrest_count","arrest_status",
                  "arrest_group_id","source_url","extraction_confidence","debug_clause"]

def finalize(df):
    out = consolidate_events(df).reset_index(drop=True)
    out.insert(0, "event_id", range(1, len(out) + 1))
    return out

def export_csv(df, path="ndlea_events.csv", keep_debug=True, force=False):
    final = finalize(df)
    validate(final, strict=not force)          # raises ValidationError unless force=True
    cols = EXPORT_COLUMNS if keep_debug else [c for c in EXPORT_COLUMNS if c != "debug_clause"]
    final[cols].to_csv(path, index=False)
    print(f"Saved {len(final)} rows -> {path}")
    return final[cols]

# Typical run:
# links = get_all_article_links(max_pages=2)
# df_raw = run_pipeline([x["url"] for x in links])
# final_df = export_csv(df_raw)

## Self-tests (run after any edit - each assertion is one rule from the spec)

In [ ]:
def _run(body, title="Test article", pub=datetime(2026, 9, 20), url="https://ndlea.gov.ng/blog/test"):
    ev = process_article({"title": title, "publication_date": pub, "body_text": body, "source_url": url})
    return build_dataframe(ev)

def run_self_tests():
    # 1. Spec example: drug<->quantity association, mg strength ignored, clause-local arrests
    df = _run("Onyeka Ofor was arrested at the Lagos airport on Wednesday 16th September 2026 with 48,900 pills of tramadol 225mg, "
              "while seven cartons of cannabis sativa weighing 59kg were intercepted with a suspect Luke Ugokwe taken into custody.")
    r = {row.drug_type: row for row in df.itertuples()}
    assert set(r) == {"tramadol", "cannabis"}, r.keys()
    assert (r["tramadol"].quantity, r["tramadol"].unit) == (48900, "pills") and pd.isna(r["tramadol"].quantity_kg)
    assert (r["cannabis"].quantity, r["cannabis"].unit, r["cannabis"].quantity_kg) == (59, "kg", 59.0)
    assert r["tramadol"].arrest_count == 1 and r["cannabis"].arrest_count == 1
    assert r["tramadol"].incident_date == "2026-09-16" and r["cannabis"].incident_date == "2026-09-16"
    assert not ((df["unit"] == "mg")).any()

    # 2. Spec example: Loud vs methamphetamine must never swap
    df = _run("In Edo State, operatives recovered 48 kilograms of Canadian Loud and 152 grams of methamphetamine.")
    r = {row.drug_type: row for row in df.itertuples()}
    assert (r["loud"].quantity, r["loud"].unit) == (48, "kilograms") and r["loud"].state == "Edo"
    assert (r["methamphetamine"].quantity, r["methamphetamine"].unit) == (152, "grams")
    assert abs(r["methamphetamine"].quantity_kg - 0.152) < 1e-9
    assert df["arrest_count"].isna().all() and (df["arrest_status"] == "Not stated").all()

    # 3. Arrest counts are NOT broadcast from an earlier sentence onto a later recovery
    df = _run("Four suspects were arrested in Kano. 48kg of cannabis was recovered from the location.")
    assert len(df) == 1 and pd.isna(df.iloc[0].arrest_count) and df.iloc[0].arrest_status == "Not stated"

    # 4. Explicit count words
    df = _run("Two suspects were arrested in Kano on Friday 18th September with 5kg of cocaine.")
    assert df.iloc[0].arrest_count == 2 and df.iloc[0].arrest_status == "Yes"
    df = _run("A suspect was arrested in Kano on Friday 18th September with 5kg of cocaine.")
    assert df.iloc[0].arrest_count == 1
    df = _run("The agency states no arrest was made after 5kg of cocaine was recovered in Kano on Friday 18th September.")
    assert df.iloc[0].arrest_status == "No" and pd.isna(df.iloc[0].arrest_count)

    # 5. Dates: never the publication date by default; future dates rejected; year inferred only if unambiguous
    df = _run("Operatives recovered 5kg of cocaine in Kano.")
    assert df.iloc[0].incident_date is None
    df = _run("Operatives recovered 5kg of cocaine in Kano on Friday 25th September.")        # after publication (20 Sep)
    assert df.iloc[0].incident_date is None
    df = _run("Operatives recovered 5kg of cocaine in Kano on Friday 18th September.")
    assert df.iloc[0].incident_date == "2026-09-18"

    # 6. Real-style multi-drug sentence -> 4 separate rows, dosage strengths ignored
    df = _run("NDLEA officers on Thursday 7th May arrested Amos Kenneth, 25, with 160.3grams of Tramadol 225mg and 100mg; "
              "80 pills of Diazepam; 38.23grams of Exol 5 and 176.93 grams of skunk.", pub=datetime(2026, 5, 10))
    got = {(r.drug_type, r.quantity, r.unit) for r in df.itertuples()}
    assert got == {("tramadol", 160.3, "grams"), ("diazepam", 80, "pills"), ("exol-5", 38.23, "grams"), ("skunk", 176.93, "grams")}, got
    assert (df["incident_date"] == "2026-05-07").all()
    assert df["arrest_group_id"].nunique() == 1              # one arrested person, not four

    # 7. Parcels + kg for the same strain -> ONE row, generic 'cannabis' descriptor ignored
    df = _run("Officers on Friday 8th May raided a Lagos mansion where 4,000 parcels of Loud, a strong strain of imported cannabis "
              "weighing 2,326 kilograms were recovered.", pub=datetime(2026, 5, 10))
    assert len(df) == 1 and df.iloc[0].drug_type == "loud" and df.iloc[0].quantity_kg == 2326.0

    # 8. 'the arrest of three suspects' (noun form) is captured
    df = _run("A search of the vehicle in Kogi State led to the recovery of 76 jumbo bags of skunk weighing 766kg and the arrest "
              "of three suspects: Mathew Omohove; Ebuka Desmond and Babangida Musa.")
    assert len(df) == 1 and df.iloc[0].arrest_count == 3

    # 9. Plausibility flag
    df = _run("Officers on Friday 18th September destroyed cannabis weighing 9,000 kilograms in Kano.")
    assert df.iloc[0].extraction_confidence in ("Medium", "Low")
    df = _run("Officers on Friday 18th September recovered cocaine weighing 9,000 kilograms in Kano.")
    assert df.iloc[0].extraction_confidence == "Low"

    # 10. Non-enforcement articles are excluded
    assert _run("NDLEA signs MoU with partners to cooperate on drug control. The agency seized 500kg of cannabis last year.",
                title="NDLEA signs MoU with foreign partner").empty
    assert _run("The chairman delivered a keynote speech at the summit.", title="Keynote speech at drug summit").empty

    # 11. Consolidation: same seizure repeated -> one row; same qty but different known dates -> two rows
    body = ("Officers on Friday 18th September arrested a suspect in Kano with 5kg of cocaine. "
            "The suspect was arrested in Kano with 5kg of cocaine.")
    df = _run(body)
    assert len(df) == 2 and len(consolidate_events(df)) == 1
    df = _run("Officers on Friday 11th September arrested a suspect in Kano with 5kg of cocaine. "
              "Officers on Friday 18th September arrested a suspect in Kano with 5kg of cocaine.")
    assert len(consolidate_events(df)) == 2

    # 12. Validation catches the error classes it is meant to catch
    bad = _run("Officers on Friday 18th September arrested a suspect in Kano with 5kg of cocaine.")
    bad2 = bad.copy(); bad2.loc[0, "arrest_count"] = 0
    assert "6_arrest_count_zero" in validate(bad2, strict=False, verbose=False)
    bad3 = bad.copy(); bad3.loc[0, "incident_date"] = "2026-10-01"
    assert "3_incident_after_publication" in validate(bad3, strict=False, verbose=False)
    bad4 = bad.copy(); bad4.loc[0, ["unit", "quantity"]] = ["mg", 225]
    assert "8_mg_strength_emitted_as_quantity" in validate(bad4, strict=False, verbose=False)
    assert validate(consolidate_events(bad), strict=False, verbose=False) == {}
    print("All self-tests passed.")

run_self_tests()